# Emotion Detection Model Training
Training emotion classifier using FER2013 dataset

In [ ]:
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

## Download Dataset
Download from: https://www.kaggle.com/datasets/msambare/fer2013

In [ ]:
# Load FER2013 dataset
train_dir = 'datasets/fer2013/train'
test_dir = 'datasets/fer2013/test'

emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

def load_emotion_data(data_dir, img_size=(48, 48)):
    images = []
    labels = []
    
    for emotion_idx, emotion in enumerate(emotion_labels):
        emotion_path = os.path.join(data_dir, emotion)
        if not os.path.exists(emotion_path):
            continue
            
        for img_name in os.listdir(emotion_path):
            img_path = os.path.join(emotion_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, img_size)
                images.append(img)
                labels.append(emotion_idx)
    
    return np.array(images), np.array(labels)

X_train, y_train = load_emotion_data(train_dir)
X_test, y_test = load_emotion_data(test_dir)

# Reshape and normalize
X_train = X_train.reshape(-1, 48, 48, 1) / 255.0
X_test = X_test.reshape(-1, 48, 48, 1) / 255.0

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Build emotion detection model
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=64
)

In [ ]:
# Save model
model.save('../models/emotion_model.h5')
print("Model saved successfully!")